# 🔎 Explore — Análise exploratória (EDA)

Projeto 7 (**Imagem**) · PAD 2026 (UFG) · fase **E (Explore)**

Objetivos desta fase:
1. Ver o **balanço das classes** (importa para o F1-macro);
2. Ver a **matriz fonte × classe** (a base do teste **LODO**);
3. **Olhar** amostras por classe e por fonte (câmeras/fundos diferentes);
4. **Testar a hipótese**: os dados agrupam por *doença* ou por *fonte/câmera*? (PCA dos *embeddings*).

> Pré-requisito: ter rodado o **Get** (`01_get.ipynb`) e salvo o dataset no Drive.

## 1. Montar o Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Carregar o dataset e o manifesto (do Drive)
Copia o derivado e o `manifest.csv` salvos pelo Get; usa só as imagens **usáveis** (sem duplicata/conflito).

In [ ]:
import pandas as pd, os, glob
os.chdir('/content')
os.makedirs('data', exist_ok=True)          # garante que data/ existe
!rm -rf data/derived
!cp -r /content/drive/MyDrive/banana_pad/derived data/derived
!cp /content/drive/MyDrive/banana_pad/manifest.csv manifest.csv
assert os.path.isdir('data/derived'), 'derived nao copiou do Drive — rode o Get (01_get) e confira a celula 14'
df = pd.read_csv('manifest.csv')
df = df[(df['dup_of'].fillna('') == '') & (df['note'].fillna('') != 'label_conflict')]
print(len(df), 'imagens usaveis |', len(glob.glob('data/derived/**/*.jpg', recursive=True)), 'arquivos')
df.head()

## 3. Balanço das classes (quão desbalanceado está?)

In [ ]:
import matplotlib.pyplot as plt
df['label'].value_counts().plot.bar(title='Imagens por classe'); plt.ylabel('n'); plt.show()

## 4. Matriz fonte × classe (a base do LODO)
Mostra quais classes existem em quais fontes — quem só aparece em 1 fonte é o ponto sensível do LODO.

In [ ]:
ct = pd.crosstab(df['source'], df['label'])
print(ct)
ct.plot.bar(stacked=True, title='Fonte x classe'); plt.ylabel('n'); plt.show()

## 5. Conferir o tamanho das imagens (devem estar ~320px)

In [ ]:
from PIL import Image
sizes = [Image.open(p).size for p in df['image_id'].sample(min(200, len(df)), random_state=0)]
print('tamanhos unicos (amostra):', sorted(set(sizes))[:8])

## 6. Amostras por classe (sanidade visual)

In [ ]:
import numpy as np
classes = sorted(df['label'].unique())
fig, ax = plt.subplots(1, len(classes), figsize=(3 * len(classes), 3))
for a, c in zip(np.atleast_1d(ax), classes):
    a.imshow(Image.open(df[df.label == c].iloc[0]['image_id'])); a.set_title(c); a.axis('off')
plt.show()

## 7. Amostras por fonte (ver a diferença de câmera/fundo)
É aqui que dá pra *ver* por que o modelo pode 'decorar' a fonte em vez da lesão.

In [ ]:
fontes = sorted(df['source'].unique())
fig, ax = plt.subplots(1, len(fontes), figsize=(3 * len(fontes), 3))
for a, s in zip(np.atleast_1d(ax), fontes):
    a.imshow(Image.open(df[df.source == s].iloc[0]['image_id'])); a.set_title(s); a.axis('off')
plt.show()

## 8. Teste da hipótese: os dados agrupam por DOENÇA ou por FONTE?
Extrai *embeddings* de uma ResNet pré-treinada, reduz a 2D com PCA e plota duas vezes:
colorido por **classe** e por **fonte**. Se os pontos se agruparem por **fonte**, é a evidência
de que o modelo tende a aprender o fundo/câmera — exatamente a hipótese do projeto.

In [ ]:
import torch, numpy as np
from torchvision import models, transforms
from sklearn.decomposition import PCA
# amostra balanceada por classe (rapido)
amostra = df.groupby('label', group_keys=False).apply(lambda g: g.sample(min(60, len(g)), random_state=0))
tf = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(),
                         transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
net = models.resnet18(weights='IMAGENET1K_V1'); net.fc = torch.nn.Identity(); net.eval().to(dev)
feats = []
with torch.no_grad():
    for p in amostra['image_id']:
        x = tf(Image.open(p).convert('RGB')).unsqueeze(0).to(dev)
        feats.append(net(x).cpu().numpy()[0])
xy = PCA(n_components=2).fit_transform(np.array(feats))

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 5))
for c in sorted(amostra['label'].unique()):
    m = (amostra['label'] == c).values; a1.scatter(xy[m, 0], xy[m, 1], s=12, label=c)
a1.set_title('Colorido por CLASSE'); a1.legend(fontsize=7)
for s in sorted(amostra['source'].unique()):
    m = (amostra['source'] == s).values; a2.scatter(xy[m, 0], xy[m, 1], s=12, label=s)
a2.set_title('Colorido por FONTE'); a2.legend(fontsize=7)
plt.show()

## 9. Conclusões (preencher com o que vocês observaram)
- Balanço das classes: _..._
- Classes de fonte única (risco no LODO): _..._
- O PCA agrupou mais por classe ou por fonte? _..._ → sustenta/refuta a hipótese.
- Próximo passo (Model): treinar e medir **F1 pooled vs LODO**.